# Temporal stability of the final ConvNeXt model on held-out GONG bursts

An earlier ResNet experiment measured frame-to-frame stability on 80 GONG bursts (same station, ±10 min, frames aligned) that no model was
trained on: every frame is segmented on its own, and every pair of frames of a burst is scored against each other
with PQ (*self-PQ*; 1 = identical instances in every frame). There, temporal distillation (`demet_tc`) made a
ResNet-34 U-Net more stable (+0.031) while plain self-training (`tekkare`) made it less stable (−0.017).

Question here: is the final ConvNeXt model, trained only on MAGFiLO, already as stable as the distilled ResNet?
All models are evaluated on the identical frames with the identical post-processing used in the earlier ResNet experiment (hysteresis
components, no scorer, 4 flips), so the differences are paired per burst.

In [ ]:
import os, sys, json, glob, itertools, time
from pathlib import Path
REPO_DIR = Path(globals().get("REPO_DIR", ".."))
GONG_DIR = Path(globals().get("GONG_DIR", "."))            # burst-building step output: kareler.csv + ayrik/*.png
CKPT = globals().get("CKPT", {})                           # name -> checkpoint path(s)
OUT_DIR = Path(globals().get("OUT_DIR", "./stability"))
DRY = os.environ.get("DRY") == "1"
sys.path.insert(0, str(REPO_DIR))
import numpy as np, pandas as pd, cv2, torch, torch.nn as nn, torch.nn.functional as F, torchvision
from filament.config import load_config, scaled
from filament.model import load_model
from filament.infer import predict
from filament import postprocess as pp, evaluate as ev
cfg = load_config(REPO_DIR / "configs" / "final.yaml")
if DRY:
    cfg = scaled(cfg, 256)
OUT_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
KR = pd.read_csv(GONG_DIR / "kareler.csv")
BURSTS = KR[KR.rol == "ayrik"].groupby("demet").apply(lambda g: list(zip(g.stem, g.temiz.astype(bool)))).to_dict()
BURSTS = {b: k for b, k in BURSTS.items() if len(k) >= 2}
print(f"{len(BURSTS)} held-out bursts, {sum(len(k) for k in BURSTS.values())} frames | device {device}")

In [ ]:
# ResNet-34 U-Net of the earlier experiments (teacher and students), unchanged
def conv_block(ci, co):
    return nn.Sequential(nn.Conv2d(ci, co, 3, padding=1, bias=False), nn.BatchNorm2d(co), nn.ReLU(True),
                         nn.Conv2d(co, co, 3, padding=1, bias=False), nn.BatchNorm2d(co), nn.ReLU(True))

class Up(nn.Module):
    def __init__(self, ci, cs, co):
        super().__init__(); self.block = conv_block(ci + cs, co)
    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        return self.block(torch.cat([x, skip], 1) if skip is not None else x)

class FilamentNet(nn.Module):
    def __init__(self):
        super().__init__()
        r = torchvision.models.resnet34(weights=None)
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu); self.pool = r.maxpool
        self.l1, self.l2, self.l3, self.l4 = r.layer1, r.layer2, r.layer3, r.layer4
        self.u4, self.u3 = Up(512, 256, 256), Up(256, 128, 128)
        self.u2, self.u1, self.u0 = Up(128, 64, 64), Up(64, 64, 32), Up(32, 0, 16)
        self.head_seg, self.head_spine = nn.Conv2d(16, 1, 1), nn.Conv2d(16, 1, 1)
    def forward(self, x):
        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)
        x0 = self.stem(x); x1 = self.l1(self.pool(x0))
        x2 = self.l2(x1); x3 = self.l3(x2); x4 = self.l4(x3)
        dd = self.u0(self.u1(self.u2(self.u3(self.u4(x4, x3), x2), x1), x0))
        return self.head_seg(dd), self.head_spine(dd)

def load_any(path):
    if "convnext" in Path(path).name:
        return load_model(path, cfg, device)
    net = FilamentNet()
    net.load_state_dict(torch.load(path, map_location="cpu"))
    return net.to(device).eval()

In [ ]:
def frame(stem):
    x = cv2.imread(str(GONG_DIR / "ayrik" / f"{stem}.png"), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
    full = cfg["data"]["full"]
    return x if x.shape[0] == full else cv2.resize(x, (full, full), interpolation=cv2.INTER_AREA)

def components(prob):
    p = cfg["postprocess"]
    return pp.components(prob, p["t_lo"], p["t_hi"], p["close"], p["min_area"])

def label_iou(la, na, lb, nb):
    if na == 0 or nb == 0:
        return np.zeros((na, nb))
    c = np.bincount(la.astype(np.int64).ravel() * (nb + 1) + lb.astype(np.int64).ravel(),
                    minlength=(na + 1) * (nb + 1)).reshape(na + 1, nb + 1)
    inter = c[1:, 1:].astype(float)
    return inter / np.maximum(c[1:, :].sum(1)[:, None] + c[:, 1:].sum(0)[None, :] - inter, 1)

def burst_stats(labs):
    """labs: [(label image, n, clean), ...] of one burst -> self-PQ parts (clean-clean, with cloudy), pair pixel IoUs,
    std and mean of the instance count."""
    p_tt, p_bi, piou = np.zeros(4), np.zeros(4), []
    for (la, na, ca), (lb, nb, cb) in itertools.combinations(labs, 2):
        part = np.array(ev.record_score(label_iou(la, na, lb, nb), np.ones(nb, bool)) if na else (0.0, 0, nb, 0))
        if ca and cb: p_tt += part
        else: p_bi += part
        ma, mb = la > 0, lb > 0
        piou.append(float((ma & mb).sum() / max((ma | mb).sum(), 1)))
    n = [k for _, k, _ in labs]
    return dict(tt=p_tt.tolist(), bi=p_bi.tolist(), piou=piou, std=float(np.std(n)), mean=float(np.mean(n)))

In [ ]:
# One burst at a time (memory stays small); every model sees the same frames. Results per burst are saved, so a
# crashed or disconnected session continues where it stopped. The ConvNeXt fold models are also averaged
# (= the test-time ensemble).
NETS = {name: load_any(path) for name, path in CKPT.items()}
FOLDS = [n for n in NETS if n.startswith("convnext_fold")]
NAMES = list(NETS) + (["convnext_4fold_mean"] if len(FOLDS) > 1 else [])
DONE_P = OUT_DIR / "per_burst.json"
DONE = json.loads(DONE_P.read_text()) if DONE_P.exists() else {}
if DONE and set(next(iter(DONE.values()))) != set(NAMES):
    print("model list changed since the saved partial run -> starting over"); DONE = {}
print(f"{len(DONE)} bursts already done")
t0 = time.time()
for i, (b, frames) in enumerate(BURSTS.items()):
    if b in DONE:
        continue
    labs = {n: [] for n in NAMES}
    for s, c in frames:
        x = frame(s)
        fold_probs = []
        for n, net in NETS.items():
            p = predict([net], x, device)
            if n in FOLDS:
                fold_probs.append(p)
            labs[n].append((*components(p), c))
        if "convnext_4fold_mean" in labs:
            labs["convnext_4fold_mean"].append((*components(np.mean(fold_probs, 0)), c))
    DONE[b] = {n: burst_stats(labs[n]) for n in NAMES}
    if len(DONE) % 5 == 0 or len(DONE) == len(BURSTS):
        tmp = DONE_P.with_suffix(".tmp"); tmp.write_text(json.dumps(DONE)); os.replace(tmp, DONE_P)
        print(f"  {len(DONE)}/{len(BURSTS)} bursts ({(time.time() - t0) / 60:.1f} min)", flush=True)
tmp = DONE_P.with_suffix(".tmp"); tmp.write_text(json.dumps(DONE)); os.replace(tmp, DONE_P)
del NETS
order = [b for b in BURSTS if b in DONE]
S = {n: dict(tt=np.array([DONE[b][n]["tt"] for b in order]), bi=np.array([DONE[b][n]["bi"] for b in order]),
             pixel_iou=float(np.mean([v for b in order for v in DONE[b][n]["piou"]])),
             count_std=float(np.mean([DONE[b][n]["std"] for b in order])),
             count_mean=float(np.mean([DONE[b][n]["mean"] for b in order]))) for n in NAMES}

In [ ]:
ref_names = [r for r in ("resnet_teacher", "resnet_demet_tc") if r in S]
rows = []
for n, r in S.items():
    row = dict(model=n, self_pq_clean=ev.pq(r["tt"]), self_pq_with_cloudy=ev.pq(r["bi"]), pixel_iou=r["pixel_iou"],
               count_std=r["count_std"], count_mean=r["count_mean"])
    for ref in ref_names:
        if ref != n:
            for k, lab in (("tt", "clean"), ("bi", "cloudy")):
                b = ev.paired_bootstrap(r[k], S[ref][k])
                row[f"vs_{ref}_{lab}"] = f"{b['diff']:+.4f} [{b['lo']:+.4f}, {b['hi']:+.4f}]"
    rows.append(row)
T = pd.DataFrame(rows)
print(T.round(4).to_string(index=False))
print("Earlier ResNet reference (same frames, earlier code): teacher 0.6344 / 0.5137, demet_tc 0.6653 / 0.5457, tekkare 0.6173 / 0.4988")
T.to_csv(OUT_DIR / "stability.csv", index=False)
np.savez(OUT_DIR / "stability_parts.npz", **{f"{n}_{k}": S[n][k] for n in S for k in ("tt", "bi")})